# Assignment 5, Question 6: Data Transformation

**Points: 20**

Transform and engineer features from the clinical trial dataset.

## Setup

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Import utilities
from q3_data_utils import load_data, clean_data, transform_types, create_bins, fill_missing

df = load_data('data/clinical_trial_raw.csv')
print(f"Loaded {len(df)} patients")

# Prewritten visualization functions for transformation analysis
def plot_distribution(series, title, figsize=(10, 6)):
    """
    Create a histogram of a numeric series.
    
    Args:
        series: pandas Series with numeric data
        title: Chart title
        figsize: Figure size tuple
    """
    plt.figure(figsize=figsize)
    series.hist(bins=30)
    plt.title(title)
    plt.xlabel('Value')
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

def plot_value_counts(series, title, figsize=(10, 6)):
    """
    Create a bar chart of value counts.
    
    Args:
        series: pandas Series with value counts
        title: Chart title
        figsize: Figure size tuple
    """
    plt.figure(figsize=figsize)
    series.plot(kind='bar')
    plt.title(title)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

'Data shape: (10000, 18)'

Loaded 10000 patients


## Part 1: Type Conversions (5 points)

1. Convert 'enrollment_date' to datetime using the `transform_types()` utility
2. Convert categorical columns ('site', 'intervention_group', 'sex') to category dtype
3. Ensure all numeric columns are proper numeric types
4. Display the updated dtypes

In [5]:
# TODO: Type conversions
# 1. Use transform_types() to convert enrollment_date to datetime
type_map = {
        'patient_id': 'string',
        'age': 'numeric',
        'bmi': 'numeric',
        'sex': 'category',
        'enrollment_date': 'datetime',
        'site': 'category',
        'systolic_bp': 'numeric',
        'diastolic_bp': 'numeric',
        'cholesterol_total': 'numeric',
        'cholesterol_hdl': 'numeric',
        'cholesterol_ldl': 'numeric',
        'glucose_fasting': 'numeric',
        'site': 'category',
        'intervention_group': 'category',
        'follow_up_months': 'numeric',
        'adverse_events': 'numeric',
        'outcome_cvd': 'string',
        'adherence_pct': 'numeric',
        'dropout': 'category'
    }
df = transform_types(df, type_map)
# 4. Display the updated dtypes using df.dtypes
display_dtypes = df.dtypes
print(display_dtypes)

patient_id            string[python]
age                            int64
sex                         category
bmi                          float64
enrollment_date       datetime64[ns]
systolic_bp                  float64
diastolic_bp                 float64
cholesterol_total            float64
cholesterol_hdl              float64
cholesterol_ldl              float64
glucose_fasting              float64
site                        category
intervention_group          category
follow_up_months               int64
adverse_events                 int64
outcome_cvd           string[python]
adherence_pct                float64
dropout                     category
dtype: object


## Part 2: Feature Engineering (8 points)

Create these new calculated columns:

1. `cholesterol_ratio` = cholesterol_ldl / cholesterol_hdl
2. `bp_category` = categorize systolic BP:
   - 'Normal': < 120
   - 'Elevated': 120-129
   - 'High': >= 130
3. `age_group` using `create_bins()` utility:
   - Bins: [0, 40, 55, 70, 100]
   - Labels: ['<40', '40-54', '55-69', '70+']
4. `bmi_category` using standard BMI categories:
   - Underweight: <18.5
   - Normal: 18.5-24.9
   - Overweight: 25-29.9
   - Obese: >=30

In [7]:
# TODO: Calculate cholesterol ratio
cholesterol_ratio = pd.Series([cholesterol_ldl / cholesterol_hdl if cholesterol_hdl != 0 else np.nan
                               for cholesterol_ldl, cholesterol_hdl in zip(df['cholesterol_ldl'], df['cholesterol_hdl'])])
df['cholesterol_ratio'] = cholesterol_ratio
print(df[['cholesterol_ldl', 'cholesterol_hdl', 'cholesterol_ratio']].head())



   cholesterol_ldl  cholesterol_hdl  cholesterol_ratio
0             41.0             55.0           0.745455
1            107.0             58.0           1.844828
2             82.0             56.0           1.464286
3            104.0             56.0           1.857143
4             75.0             78.0           0.961538


In [10]:
# TODO: Categorize blood pressure
bp_category = pd.Series([0, 120, 129, 130, 200])
bp_category = pd.cut(df['systolic_bp'], bins=[0, 120, 130, 200], labels=['Normal', 'Elevated', 'High'])
print(bp_category)  # [Normal, Elevated, High]


0       Elevated
1           High
2       Elevated
3         Normal
4         Normal
          ...   
9995    Elevated
9996    Elevated
9997      Normal
9998        High
9999        High
Name: systolic_bp, Length: 10000, dtype: category
Categories (3, object): ['Normal' < 'Elevated' < 'High']


**Note:** The `create_bins()` function has an optional `new_column` parameter. If you don't specify it, the new column will be named `{original_column}_binned`. You can use `new_column='age_group'` to give it a custom name.


In [3]:
# TODO: Create age groups
df = create_bins(df, 'age', bins=[0, 40, 55, 70, 100], labels=['<40', '40-54', '55-69', '70+'], new_column='age_group')
print(df['age_group'].value_counts())

age_group
70+      7333
55-69    2204
40-54     263
<40         0
Name: count, dtype: int64


In [10]:
# TODO: Create BMI categories
bmi_category = pd.Series([0, 18.5, 24.9, 29.9, 34.9, 39.9, 100])
bmi_category = pd.cut(df['bmi'], bins = [0, 18.5, 25, 30, 100], labels=['Underweight', 'Normal', 'Overweight', 'Obese'])
print(bmi_category)  # [Underweight, Normal, Overweight, Obesity I, Obesity II, Obesity III]

0       Overweight
1              NaN
2              NaN
3       Overweight
4              NaN
           ...    
9995        Normal
9996    Overweight
9997        Normal
9998    Overweight
9999    Overweight
Name: bmi, Length: 10000, dtype: category
Categories (4, object): ['Underweight' < 'Normal' < 'Overweight' < 'Obese']


## Part 3: String Cleaning (2 points)

If there are any string columns that need cleaning:
1. Convert to lowercase
2. Strip whitespace
3. Replace any placeholder values

In [15]:
# TODO: String cleaning
display(df.dtypes)
string_columns = df.select_dtypes(include='string').columns
for f in string_columns:
    df[f] = df[f].str.lower().strip()
    print(f"Cleaned string column: {f}")
    print(df[f].value_counts())


patient_id             object
age                     int64
sex                    object
bmi                   float64
enrollment_date        object
systolic_bp           float64
diastolic_bp          float64
cholesterol_total     float64
cholesterol_hdl       float64
cholesterol_ldl       float64
glucose_fasting       float64
site                   object
intervention_group     object
follow_up_months        int64
adverse_events          int64
outcome_cvd            object
adherence_pct         float64
dropout                object
dtype: object

## Part 4: One-Hot Encoding (5 points)

Create dummy variables for categorical columns:
1. One-hot encode 'intervention_group' using `pd.get_dummies()`
2. One-hot encode 'site'
3. Drop the original categorical columns
4. Show the new shape and column names

In [21]:
# TODO: One-hot encoding
df = pd.get_dummies(df, columns=['intervention_group', 'site'])
print(df.shape)
print(df.columns)



(10000, 76)
Index(['patient_id', 'age', 'sex', 'bmi', 'enrollment_date', 'systolic_bp',
       'diastolic_bp', 'cholesterol_total', 'cholesterol_hdl',
       'cholesterol_ldl', 'glucose_fasting', 'follow_up_months',
       'adverse_events', 'outcome_cvd', 'adherence_pct', 'dropout',
       'intervention_group_  CONTROL  ', 'intervention_group_  Contrl  ',
       'intervention_group_  Control  ', 'intervention_group_  TREATMENT A  ',
       'intervention_group_  TREATMENT B  ',
       'intervention_group_  Treatmen A  ',
       'intervention_group_  Treatment  B  ',
       'intervention_group_  Treatment A  ',
       'intervention_group_  Treatment B  ',
       'intervention_group_  TreatmentA  ', 'intervention_group_  control  ',
       'intervention_group_  treatment a  ',
       'intervention_group_  treatment b  ', 'intervention_group_CONTROL',
       'intervention_group_Contrl', 'intervention_group_Control',
       'intervention_group_TREATMENT A', 'intervention_group_TREATMENT B',

## Part 5: Save Transformed Data

Save the fully transformed dataset to `output/q6_transformed_data.csv`

In [25]:
# TODO: Save transformed data
# df_transformed.to_csv('output/q6_transformed_data.csv', index=False)
df_transformed = df
df_transformed.to_csv('output/q5_transformed_data.csv', index=False)